Install necessary packages

In [32]:
%pip install -U bitsandbytes huggingface_hub transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA H200 NVL


In [35]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  38158 MiB |  39514 MiB | 117301 GiB | 117264 GiB |
|       from large pool |  37958 MiB |  39313 MiB | 113953 GiB | 113916 GiB |
|       from small pool |    200 MiB |    361 MiB |   3348 GiB |   3347 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  38158 MiB |  39514 MiB | 117301 GiB | 117264 GiB |
|       from large pool |  37958 MiB |  39313 MiB | 113953 GiB |

In [ ]:
#initalizing the model with 4 bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.1-70B-Instruct"

# Configure 4-bit quantization to save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Step #0, Prompting the model for tables and saving them in csv form

In [ ]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
prompt0 = [
    {
        "role": "system",
        "content": "You are a careful researcher who generates clean, realistic synthetic statistical tables."
    },
    {
        "role": "user",
        "content": """Generate exactly one synthetic statistical table in valid CSV format.

Requirements:
1. The table must have between 5 and 10 columns.
2. The table must contain exactly 15 data rows, plus 1 header row.
3. Invent realistic column names. Include a mix of:
   - numeric columns (integers and/or floats)
   - categorical columns
4. Make the data look plausible for a real-world dataset.
5. Ensure values are varied across rows.
6. All rows must have the same number of columns as the header.
7. Do not include commas inside cell values.
8. Do not include explanations, markdown, code fences, titles, or any text before or after the CSV.
9. Output only the CSV table.

The dataset can be about any realistic domain such as health, education, business, transportation, or demographics."""
    },
]

In [ ]:
from transformers import GenerationConfig

prompt = tokenizer.apply_chat_template(
    prompt0,
    tokenize=False,
    add_generation_prompt=True
)

gen_config = GenerationConfig(
    do_sample=False,
    temperature=1.0,
    top_p=1.0,
    max_new_tokens=1000,
    eos_token_id=None
)

generation = generator(
    prompt0,
    generation_config=gen_config
)

print(f"Generation: {generation[0]['generated_text']}")

In [ ]:
# Get the assistant message from generated_text
assistant_output = generation[-1]['generated_text'][-1]  # last item
full_text = assistant_output['content']  # this is your CSV string

# Preview
print(full_text)

In [ ]:
import os

# Folder where CSVs will be saved
folder_path = "LLAMA/a.LLM_Tables"
os.makedirs(folder_path, exist_ok=True)

# Split the full string into candidate tables
# We assume that each table starts with a header row (quotes optional)
candidate_tables = full_text.strip().split("\n\n")
#print(candidate_tables)

table_counter = 1

for table_text in candidate_tables:
    # Remove any trailing "assistant" or blank lines
    lines = [line for line in table_text.strip().split("\n") if line.strip() and line.strip().lower() != "assistant"]

    if not lines:
        continue  # skip empty sections

    header = lines[0]

    rows = lines[1:]


    # Check for incomplete table: all rows must have same number of columns as header
    num_cols = len(header.split(","))
    valid_rows = [r for r in rows if len(r.split(",")) == num_cols]

    if not valid_rows:
        continue  # skip tables with no valid rows

    # Reconstruct the table
    table_cleaned = "\n".join([header] + valid_rows)

    # Save to CSV
    file_name = f"table_{table_counter}.csv"
    full_path = os.path.join(folder_path, file_name)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(table_cleaned)

    print(f"Saved {file_name} with {len(valid_rows)} rows")
    table_counter += 1

Step #1, Prompting the model for Statments INPUT READING and GENERATION OF STATEMENTS

In [ ]:

# Folder where the tables were saved
folder_path = "LLAMA/a.LLM_Tables"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
csv_files.sort()  # optional: ensure consistent order

for csv_file in csv_files:
    full_path = os.path.join(folder_path, csv_file)

    # Read the table
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{csv_file} is empty, skipping")
        continue

    # Extract header and rows
    header = lines[0]
    rows = lines[1:]

    print(f"Processing {csv_file}: {len(rows)} rows")
    ##PROVIDE EXAMPLES OF WHAT KIND OF STATEMENTS TO GENERATE

    prompt1 = [
    {
        "role": "system",
        "content": "You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables."
    },
    {
        "role": "user",
        "content": f"""
Read the following table carefully:

{table_text}

Your task is to generate exactly 15 logical statements that are TRUE of this table.

Definition:
A statement is INTERESTING if it reveals a non-obvious relationship, constraint, subgroup pattern, interaction between variables, or exception that cannot be seen from a single column alone.

Requirements:
1. Every statement must be factually true.
2. Each statement must reveal a meaningful pattern in the data.
3. Avoid trivial or obvious statements.
4. Do NOT restate individual rows.
5. Do NOT produce simple existence statements like:
   - "There exists a row where Country is X"
6. Do NOT produce weak majority statements like:
   - "Most rows have Age > 25"
7. Avoid statements that involve only ONE column.

Structure requirements:
8. At least 10 statements must involve at least TWO columns.
9. At least 5 statements must involve THREE or more columns.
10. At least 5 statements must use conditional (if-then) logic.
11. At least 3 statements must describe subgroup-specific patterns.
12. At least 2 statements must describe exceptions or constraints.

Preferred types of insights:
- Relationships between categories and numeric ranges
- Constraints linking multiple variables
- Patterns that only hold within specific subgroups
- Interactions (e.g., JobType + Department + Salary)
- Conditional dependencies across columns
- Non-obvious restrictions or correlations

Good examples:
- All rows in subgroup X with condition Y also satisfy Z.
- If a row has A and B, then C must also hold.
- Every high-value case belongs to a specific subset of categories.
- Rows with condition X only appear alongside condition Y.
- All exceptions to a broader trend belong to the same subgroup.

Bad examples:
- "There exists a row where City is New York"
- "Most rows have Salary > 50000"
- "All Finance rows have Salary > 60000" (unless it reveals something deeper)
- Any statement based on a single column only

Diversity:
13. Use a mix of:
    - universal statements
    - conditional statements
    - subgroup patterns
    - constrained patterns
14. Avoid repeating the same structure across all statements.

Output format:
15. Number statements 1 through 15.
16. Output only the statements, one per line.
17. Do NOT include explanations.

Final check:
Before outputting each statement, verify:
- it is true
- it involves multiple columns (unless absolutely necessary)
- it would be considered non-trivial by a human analyst

Return exactly 15 statements.
"""
    },
]
    
    #Generate the statements necessary
    generation = generator(
    prompt1,
    do_sample=False,
    temperature=1.0,
    top_p=1,
    max_new_tokens=10000,
    eos_token_id=tokenizer.eos_token_id)
    print(f"Generation: {generation[0]['generated_text']}")
    # Get the assistant message from generated_text
    statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
    statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
    # Preview
    print(statements_LLM_output_text[1000:2000])
    #save the output of the LLM generated tasks
    file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"
    folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"
    full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)

    with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
      f.write(statements_LLM_output_text)
    print(f"Saved {file_name_LLM}.")


STEP2: GENERATING A PYTHON CODE THAT WOULD CHECK TRUTH/FALSITY OF THE ABOVE STATEMENTS AND ALSO PROVIDE AN EXPLANATION FOR THAT

In [ ]:
import re
import subprocess
import pandas as pd

# folder paths
folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"
folder_path               = "LLAMA/a.LLM_Tables"
folder_path_python_code   = "LLAMA/c.checking_statements"
folder_path_results = "LLAMA/d.checking_statements_output"

# helper to extract
def extract_python_code(raw: str) -> str:
    """
    Strip markdown fences if present (```python ... ``` or ``` ... ```).
    Falls back to returning the full string if no fences are found.
    """
    # Match ```python ... ``` or ``` ... ``` (non-greedy, dotall)
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    # No fences found — return as-is (model already output plain code)
    return raw.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    for csv_file in csv_files:
        if csv_file[:-4] not in LLM_statement_text_file:
            continue

        full_csv_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements {lines} and the header names of the table {df.head(0)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""},
        ]

        # ── Generate ───────────────────────────────────────────────────────────
        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]

        # ── Clean: strip markdown fences reliably ──────────────────────────────
        python_code = extract_python_code(raw_code)

        # ── Save ───────────────────────────────────────────────────────────────
        python_file_name_LLM = f"python_code_{csv_file[:-4]}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        # ── Execute automatically ──────────────────────────────────────────────
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_llama_inferences.txt"
        full_path_results_file = os.path.join(folder_path_results, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")